<a href="https://colab.research.google.com/github/lauraandreea126/PMP-2026/blob/main/Lab04/Tema3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tema 3 – Laborator 4
## Retele Bayesiene

In aceasta tema vom rezolva probleme folosind retele Bayesiene. Vom defini variabilele aleatoare, vom reprezenta dependentele dintre ele printr-un graf orientat aciclic si vom folosi probabilitati conditionate pentru a descrie modelul.

Pentru calcule vom folosi biblioteca `pgmpy`, care permite construirea retelelor Bayesiene si efectuarea inferentei.



## Exercitiul 1 – Problema Monty Hall

Problema Monty Hall este un exemplu de problema probabilistica in care informatia observata modifica probabilitatea evenimentelor.

Un concurent alege una dintre cele trei usi. In spatele unei usi se afla premiul, iar in spatele celorlalte doua usi nu se afla premiul.

Dupa alegerea concurentului, prezentatorul deschide o alta usa, stiind unde se afla premiul si alegand o usa in spatele careia nu se afla premiul. Concurentul poate pastra usa initiala sau poate schimba alegerea.

Vrem sa modelam aceasta situatie cu ajutorul unei retele Bayesiene si sa determinam cum se modifica probabilitatea ca premiul sa se afle in spatele unei anumite usi dupa ce observam usa deschisa de prezentator.

### Definirea variabilelor

Vom folosi trei variabile aleatoare:

- `Guest` – usa aleasa initial de concurent.
- `Prize` – usa in spatele careia se afla premiul.
- `Monty` – usa deschisa de prezentator.

Fiecare variabila poate avea valorile 0, 1 sau 2, care reprezinta cele trei usi.

Presupunem ca alegerea initiala a concurentului este uniforma si ca premiul se afla cu aceeasi probabilitate in spatele fiecarei usi.

Prezentatorul nu deschide usa aleasa de concurent si nici usa in spatele careia se afla premiul. Daca are doua usi pe care le poate deschide, presupunem ca le alege cu probabilitati egale.

In [1]:
!pip install -q pgmpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.5 MB/s eta 0:00:00


### Construirea retelei Bayesiene

Pentru a construi reteaua Bayesiana, trebuie sa stabilim dependentele dintre variabile.

Premiul (`Prize`) si alegerea initiala a concurentului (`Guest`) sunt independente, deoarece alegerea concurentului nu depinde de pozitia premiului.

Usa deschisa de prezentator (`Monty`) depinde de alegerea concurentului si de pozitia premiului. Prezentatorul trebuie sa tina cont de aceste doua informatii pentru a alege o usa pe care nu se afla premiul si care nu a fost aleasa de concurent.

Prin urmare, vom folosi urmatoarele legaturi:

- `Guest` → `Monty`
- `Prize` → `Monty`



In [2]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

model = DiscreteBayesianNetwork([
    ("Guest", "Monty"),
    ("Prize", "Monty")
])

/usr/local/lib/python3.13/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in v1.3.0. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


### Probabilitatile initiale

Presupunem ca toate cele trei usi au aceeasi probabilitate de a fi alese de concurent si aceeasi probabilitate de a ascunde premiul.

Prin urmare, pentru variabila `Guest`, avem:

$$
P(Guest=0)=P(Guest=1)=P(Guest=2)=\frac{1}{3}
$$

La fel, pentru variabila `Prize`, avem:

$$
P(Prize=0)=P(Prize=1)=P(Prize=2)=\frac{1}{3}
$$

Suma probabilitatilor pentru fiecare variabila este:

$$
\frac{1}{3}+\frac{1}{3}+\frac{1}{3}=1
$$

Asadar, distributiile initiale sunt uniforme.

In [5]:
cpd_guest = TabularCPD(
    variable="Guest",
    variable_card=3,
    values=[[1/3], [1/3], [1/3]]
)

cpd_prize = TabularCPD(
    variable="Prize",
    variable_card=3,
    values=[[1/3], [1/3], [1/3]]
)

### Probabilitatile conditionate pentru Monty

Variabila `Monty` depinde de valorile variabilelor `Guest` si `Prize`.

Prezentatorul nu poate deschide usa aleasa de concurent si nici usa care ascunde premiul.

Daca ramane o singura usa posibila, probabilitatea ca aceasta sa fie deschisa este 1.

Daca exista doua usi posibile, presupunem ca prezentatorul alege fiecare usa cu probabilitatea 1/2.

Daca usa aleasa de concurent coincide cu usa in spatele careia se afla premiul, prezentatorul poate deschide oricare dintre celelalte doua usi, fiecare cu probabilitatea 1/2.

In [7]:
cpd_monty = TabularCPD(
    variable="Monty",
    variable_card=3,
    values=[
        [0,   0,   0,   0,   0.5, 1,   0,   1,   0.5],
        [0.5, 0,   1,   0,   0,   0,   1,   0,   0.5],
        [0.5, 1,   0,   1,   0.5, 0,   0,   0,   0]
    ],
    evidence=["Guest", "Prize"],
    evidence_card=[3, 3]
)

### Verificarea modelului

Dupa definirea probabilitatilor initiale si a probabilitatilor conditionate, adaugam tabelele in reteaua Bayesiana.

Folosim metoda `check_model()` pentru a verifica daca fiecare variabila are tabelul de probabilitati definit corect si daca probabilitatile sunt valide.

Daca verificarea este reusita, putem folosi reteaua pentru a calcula probabilitati conditionate.

In [8]:
model.add_cpds(cpd_guest, cpd_prize, cpd_monty)

print("Modelul este corect:", model.check_model())

Modelul este corect: True


### Inferenta Bayesiana

Dupa ce prezentatorul deschide o usa, putem folosi aceasta informatie pentru a actualiza probabilitatile.

Presupunem ca participantul a ales initial usa 0, iar prezentatorul a deschis usa 2.

Vrem sa calculam probabilitatea ca premiul sa se afle in spatele fiecarei usi, stiind aceste informatii.

Vom calcula:

P(Prize | Guest = 0, Monty = 2)

Folosim reteaua Bayesiana definita anterior si metoda `VariableElimination` pentru a obtine distributia conditionata.

In [9]:
infer = VariableElimination(model)

rezultat = infer.query(
    variables=["Prize"],
    evidence={"Guest": 0, "Monty": 2}
)

print(rezultat)

+----------+--------------+
| Prize    |   phi(Prize) |
+==========+==============+
| Prize(0) |       0.3333 |
+----------+--------------+
| Prize(1) |       0.6667 |
+----------+--------------+
| Prize(2) |       0.0000 |
+----------+--------------+


### Interpretarea rezultatului

Am stabilit ca participantul a ales usa 0, iar prezentatorul a deschis usa 2.

Din rezultatul inferentei obtinem probabilitatile conditionate pentru variabila `Prize`.

Usa 0 este usa aleasa initial de participant. Usa 1 este singura usa inchisa, in afara de cea aleasa initial.

Daca premiul se afla in spatele usii 0, prezentatorul putea deschide usa 1 sau usa 2. Daca premiul se afla in spatele usii 1, prezentatorul era obligat sa deschida usa 2.

Prin urmare, dupa observarea usii deschise, probabilitatea ca premiul sa fie in spatele usii 1 este mai mare decat probabilitatea ca premiul sa fie in spatele usii 0.

Acest rezultat arata de ce schimbarea usii poate fi avantajoasa in problema Monty Hall.

### Calculul probabilitatilor

Stim ca participantul a ales usa 0, iar prezentatorul a deschis usa 2.

Vrem sa calculam:

$$
P(Prize=1\mid Guest=0,Monty=2)
$$

Folosim formula probabilitatii conditionate:

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
$$

In cazul nostru:

$$
P(Prize=1\mid Guest=0,Monty=2)
=\frac{P(Prize=1,Guest=0,Monty=2)}{P(Guest=0,Monty=2)}
$$

**Numaratorul:**

$$
\begin{aligned}
&P(Prize=1)\cdot P(Guest=0)\\
&\qquad\cdot P(Monty=2\mid Guest=0,Prize=1)\\
&=\frac13\cdot\frac13\cdot1=\frac19
\end{aligned}
$$

**Numitorul** se obtine adunand probabilitatile cazurilor posibile:

$$
\begin{aligned}
P(Guest=0,Monty=2)
&=P(Prize=0,Guest=0,Monty=2)\\
&\quad+P(Prize=1,Guest=0,Monty=2)\\
&\quad+P(Prize=2,Guest=0,Monty=2)
\end{aligned}
$$

$$
\begin{aligned}
P(Guest=0,Monty=2)
&=\frac13\cdot\frac13\cdot\frac12\\
&\quad+\frac13\cdot\frac13\cdot1\\
&\quad+\frac13\cdot\frac13\cdot0\\
&=\frac1{18}+\frac19+0\\
&=\frac16
\end{aligned}
$$

Prin urmare:

$$
\begin{aligned}
P(Prize=1\mid Guest=0,Monty=2)
&=\frac{1/9}{1/6}\\
&=\frac19\cdot6\\
&=\frac23
\end{aligned}
$$

Asadar, probabilitatea ca premiul sa fie in spatele usii 1 este:

$$
\boxed{\frac23\approx66.67\%}
$$

## Exercitiul 2 – Retea Bayesiana pentru diagnosticul unei boli pulmonare

Un medic doreste sa determine probabilitatea ca un pacient sa aiba o boala pulmonara, folosind informatii despre simptomele si investigatiile pacientului.

Vom folosi urmatoarele variabile aleatoare:

- `B` – prezenta bolii pulmonare;
- `T` – rezultatul testului;
- `X` – rezultatul radiografiei;
- `D` – dificultatea de respiratie.

Variabilele pot avea valorile 0 si 1:

- `B = 1` inseamna ca pacientul are boala, iar `B = 0` inseamna ca nu are boala;
- `T = 1` inseamna ca testul este pozitiv, iar `T = 0` inseamna ca testul este negativ;
- `X = 1` inseamna ca radiografia este anormala, iar `X = 0` inseamna ca radiografia este normala;
- `D = 1` inseamna ca pacientul are dificultati de respiratie, iar `D = 0` inseamna ca nu are dificultati de respiratie.

Scopul este sa construim reteaua Bayesiana si sa calculam probabilitatile cerute folosind informatiile din enunt.

### Construirea retelei Bayesiene

Boala pulmonara (`B`) este variabila de care depind rezultatul testului (`T`) si rezultatul radiografiei (`X`).

Dificultatea de respiratie (`D`) depinde de prezenta bolii si de rezultatul testului.

Prin urmare, reteaua Bayesiana va avea urmatoarele legaturi:

- `B` → `T`
- `B` → `X`
- `B` → `D`
- `T` → `D`

Astfel, putem reprezenta dependentele dintre variabile si putem folosi probabilitatile conditionate din enunt pentru a defini modelul.

In [10]:
model2 = DiscreteBayesianNetwork([
    ("B", "T"),
    ("B", "X"),
    ("B", "D"),
    ("T", "D")
])

### Definirea probabilitatilor

Din enunt avem urmatoarele probabilitati:

* text cursiv1. Probabilitatea initiala a bolii pulmonare:*

$$
P(B = 1) = 0.1
$$

$$
P(B = 0) = 0.9
$$

*2. Probabilitatea conditionata pentru testul T:*

$$
P(T = 1 \mid B = 1) = 0.8
$$

$$
P(T = 1 \mid B = 0) = 0.3
$$

*3. Probabilitatea conditionata pentru radiografia X:*

$$
P(X = 1 \mid B = 1) = 0.9
$$

$$
P(X = 1 \mid B = 0) = 0.1
$$

*4. Probabilitatea conditionata pentru dificultatea de respiratie D:*

$$
P(D = 1 \mid B = 1, T = 1) = 0.9
$$

$$
P(D = 1 \mid B = 1, T = 0) = 0.4
$$

$$
P(D = 1 \mid B = 0, T = 1) = 0.5
$$

$$
P(D = 1 \mid B = 0, T = 0) = 0.1
$$

Pentru fiecare variabila binara, probabilitatea valorii 0 se obtine scazand din 1 probabilitatea valorii 1.

De exemplu:

$$
P(T = 0 \mid B = 1) = 1 - 0.8 = 0.2
$$

$$
P(X = 0 \mid B = 1) = 1 - 0.9 = 0.1
$$

$$
P(D = 0 \mid B = 1, T = 1) = 1 - 0.9 = 0.1
$$

In [29]:
cpd_B = TabularCPD(
    variable="B",
    variable_card=2,
    values=[[0.9], [0.1]]
)

cpd_T = TabularCPD(
    variable="T",
    variable_card=2,
    values=[
        [0.7, 0.2],
        [0.3, 0.8]
    ],
    evidence=["B"],
    evidence_card=[2]
)

cpd_X = TabularCPD(
    variable="X",
    variable_card=2,
    values=[
        [0.9, 0.1],
        [0.1, 0.9]
    ],
    evidence=["B"],
    evidence_card=[2]
)

cpd_D = TabularCPD(
    variable="D",
    variable_card=2,
    values=[
        [0.9, 0.5, 0.6, 0.1],
        [0.1, 0.5, 0.4, 0.9]
    ],
    evidence=["B", "T"],
    evidence_card=[2, 2]
)

In [30]:
model2.add_cpds(cpd_B, cpd_T, cpd_X, cpd_D)

print("Modelul este corect:", model2.check_model())

Modelul este corect: True


### a) Calculul probabilitatii ca pacientul sa aiba boala pulmonara

Se observa ca pacientul are dificultati de respiratie, adica \(D = 1\).

Trebuie sa determinam:

$$
P(B = 1 \mid D = 1)
$$

Folosim formula probabilitatii conditionate:

$$
P(B = 1 \mid D = 1)
= \frac{P(B = 1, D = 1)}{P(D = 1)}
$$

Pentru a calcula aceasta probabilitate, avem nevoie de probabilitatea comuna \(P(B = 1, D = 1)\) si de probabilitatea \(P(D = 1)\).

Deoarece \(D\) depinde atat de \(B\), cat si de \(T\), trebuie sa luam in considerare ambele valori posibile ale testului \(T\): \(T = 0\) si \(T = 1\).

### Calculul probabilitatilor

Mai intai calculam \(P(B = 1, D = 1)\).

Testul \(T\) poate avea doua valori: 0 sau 1. Prin urmare, impartim calculul in doua cazuri.

**Cazul 1: \(T = 1\)**

$$\begin{aligned}
P(B = 1, T = 1, D = 1)
&= P(B = 1) \cdot P(T = 1 \mid B = 1) \cdot P(D = 1 \mid B = 1, T = 1)\\
&= 0.1 \cdot 0.8 \cdot 0.9\\
&= 0.072
\end{aligned}$$

**Cazul 2: \(T = 0\)**

$$\begin{aligned}
P(B = 1, T = 0, D = 1)
&= P(B = 1) \cdot P(T = 0 \mid B = 1) \cdot P(D = 1 \mid B = 1, T = 0)\\
&= 0.1 \cdot 0.2 \cdot 0.4\\
&= 0.008
\end{aligned}$$

Adunam cele doua cazuri:

$$\begin{aligned}
P(B = 1, D = 1)
&= 0.072 + 0.008\\
&= 0.080
\end{aligned}$$

Acum calculam \(P(B = 0, D = 1)\).

**Cazul 1: \(T = 1\)**

$$\begin{aligned}
P(B = 0, T = 1, D = 1)
&= 0.9 \cdot 0.3 \cdot 0.5\\
&= 0.135
\end{aligned}$$

**Cazul 2: \(T = 0\)**

$$\begin{aligned}
P(B = 0, T = 0, D = 1)
&= 0.9 \cdot 0.7 \cdot 0.1\\
&= 0.063
\end{aligned}$$

Adunam cele doua cazuri:

$$\begin{aligned}
P(B = 0, D = 1)
&= 0.135 + 0.063\\
&= 0.198
\end{aligned}$$

Pentru a calcula \(P(D = 1)\), adunam probabilitatile celor doua situatii posibile pentru \(B\):

$$P(D = 1) = P(B = 1, D = 1) + P(B = 0, D = 1)$$

$$\begin{aligned}
P(D = 1) &= 0.080 + 0.198\\
&= 0.278
\end{aligned}$$

Aplicam formula probabilitatii conditionate:

$$P(B = 1 \mid D = 1) = \frac{P(B = 1, D = 1)}{P(D = 1)}$$

$$\begin{aligned}
P(B = 1 \mid D = 1)
&= \frac{0.080}{0.278}\\
&\approx 0.2878
\end{aligned}$$

Asadar, probabilitatea ca pacientul sa aiba boala pulmonara, stiind ca are dificultati de respiratie, este de aproximativ **28.78%**.

In [31]:
infer2 = VariableElimination(model2)

rezultat_a = infer2.query(
    variables=["B"],
    evidence={"D": 1}
)

print(rezultat_a)

+------+----------+
| B    |   phi(B) |
+======+==========+
| B(0) |   0.7122 |
+------+----------+
| B(1) |   0.2878 |
+------+----------+


### Concluzie

Dupa ce am observat ca pacientul are dificultati de respiratie, probabilitatea ca acesta sa aiba boala pulmonara este de aproximativ 28.78%.

Probabilitatea este mai mare decat probabilitatea initiala de 10%, deoarece dificultatea de respiratie este mai probabila in cazul pacientilor care au boala.

Am obtinut rezultatul atat prin calcul manual, folosind probabilitatea conditionata si impartirea in cazuri, cat si prin inferenta cu reteaua Bayesiana definita in Python.

### b) Calculul probabilitatii ca pacientul sa aiba boala pulmonara

Medicul observa ca pacientul are tuse si dificultati de respiratie.

Prin urmare, stim ca: \(T = 1\) si \(D = 1\).

Trebuie sa determinam:

$$
P(B = 1 \mid T = 1, D = 1)
$$

Folosim formula probabilitatii conditionate:

$$
P(B = 1 \mid T = 1, D = 1)
= \frac{P(B = 1, T = 1, D = 1)}{P(T = 1, D = 1)}
$$

Pentru a calcula numitorul, impartim situatia in doua cazuri:

- pacientul are boala pulmonara, \(B = 1\);
- pacientul nu are boala pulmonara, \(B = 0\).

Calculam probabilitatea pentru fiecare caz si apoi le adunam.

### Calculul probabilitatilor

Mai intai calculam numaratorul:

$$P(B = 1, T = 1, D = 1)$$

$$\begin{aligned} &= P(B = 1) \cdot P(T = 1 \mid B = 1) \cdot P(D = 1 \mid B = 1, T = 1)\\ &= 0.1 \cdot 0.8 \cdot 0.9\\ &= 0.072 \end{aligned}$$

Acum calculam probabilitatea pentru cazul in care pacientul nu are boala:

$$P(B = 0, T = 1, D = 1)$$

$$\begin{aligned} &= P(B = 0) \cdot P(T = 1 \mid B = 0) \cdot P(D = 1 \mid B = 0, T = 1)\\ &= 0.9 \cdot 0.3 \cdot 0.5\\ &= 0.135 \end{aligned}$$

Adunam cele doua cazuri pentru a obtine numitorul:

$$\begin{aligned} P(T = 1, D = 1) &= P(B = 1, T = 1, D = 1) + P(B = 0, T = 1, D = 1)\\ &= 0.072 + 0.135\\ &= 0.207 \end{aligned}$$

Aplicam formula probabilitatii conditionate:

$$P(B = 1 \mid T = 1, D = 1) = \frac{P(B = 1, T = 1, D = 1)}{P(T = 1, D = 1)}$$

$$\begin{aligned} P(B = 1 \mid T = 1, D = 1) &= \frac{0.072}{0.207}\\ &\approx 0.3478 \end{aligned}$$

Asadar, probabilitatea ca pacientul sa aiba boala pulmonara, stiind ca are tuse si dificultati de respiratie, este de **34.78%**.

In [32]:
rezultat_b = infer2.query(
    variables=["B"],
    evidence={"T": 1, "D": 1}
)

print(rezultat_b)

+------+----------+
| B    |   phi(B) |
+======+==========+
| B(0) |   0.6522 |
+------+----------+
| B(1) |   0.3478 |
+------+----------+


### Concluzie

Dupa ce medicul observa ca pacientul are tuse si dificultati de respiratie, probabilitatea ca pacientul sa aiba boala pulmonara este de **34.78%**.

Aceasta valoare este mai mare decat probabilitatea initiala a bolii, care era de 10%.

Am obtinut rezultatul folosind formula probabilitatii conditionate si am verificat calculul cu ajutorul inferentei Bayesiene din `pgmpy`.

### c) Probabilitatea ca radiografia sa fie anormala

Stim ca pacientul nu are boala pulmonara, adica \(B = 0\).

Trebuie sa determinam probabilitatea ca radiografia sa fie anormala:

$$
P(X = 1 \mid B = 0)
$$

Din enuntul problemei avem:

$$
P(X = 1 \mid B = 0) = 0.1
$$

Aceasta probabilitate reprezinta sansa ca radiografia sa fie anormala atunci cand pacientul nu are boala pulmonara.

Probabilitatea ca radiografia sa fie normala se calculeaza astfel:

$$
P(X = 0 \mid B = 0) = 1 - P(X = 1 \mid B = 0)
$$

$$
P(X = 0 \mid B = 0) = 1 - 0.1
$$

$$
P(X = 0 \mid B = 0) = 0.9
$$

Verificam suma probabilitatilor:

$$
P(X = 0 \mid B = 0) + P(X = 1 \mid B = 0)
$$

$$
\begin{aligned}
&= 0.9 + 0.1\\
&= 1
\end{aligned}
$$

In [33]:
rezultat_c = infer2.query(
    variables=["X"],
    evidence={"B": 0}
)

print(rezultat_c)

+------+----------+
| X    |   phi(X) |
+======+==========+
| X(0) |   0.9000 |
+------+----------+
| X(1) |   0.1000 |
+------+----------+


### Concluzie

Probabilitatea ca radiografia sa fie anormala, stiind ca pacientul nu are boala pulmonara, este de 0.1, adica 10%.

Probabilitatea ca radiografia sa fie normala este de 0.9, adica 90%.

Am obtinut aceste valori din tabelul probabilitatilor conditionate pentru variabila X si am verificat rezultatul folosind inferenta Bayesiana in Python.

### d) Calculul probabilitatilor folosind regula lui Bayes

**1. Probabilitatea ca pacientul sa aiba boala, stiind ca are dificultati de respiratie**

Aplicam regula lui Bayes:

$$P(B=1\mid D=1)=\frac{P(B=1,D=1)}{P(D=1)}$$

Calculam probabilitatea comuna pentru pacientii care au boala si dificultati de respiratie:

$$P(B=1,D=1)=P(B=1)\cdot P(T=1\mid B=1)\cdot P(D=1\mid B=1,T=1) + P(B=1)\cdot P(T=0\mid B=1)\cdot P(D=1\mid B=1,T=0)$$

Inlocuim valorile:

$$P(B=1,D=1)=0.1\cdot0.8\cdot0.9 + 0.1\cdot0.2\cdot0.4$$

$$P(B=1,D=1)=0.072 + 0.008 = 0.080$$

Pentru pacientii fara boala:

$$P(B=0,D=1)=0.9\cdot0.3\cdot0.5 + 0.9\cdot0.7\cdot0.1$$

$$P(B=0,D=1)=0.135 + 0.063 = 0.198$$

Probabilitatea totala de a avea dificultati de respiratie este:

$$P(D=1)=0.080 + 0.198 = 0.278$$

Aplicam regula lui Bayes:

$$P(B=1\mid D=1)=\frac{0.080}{0.278}\approx0.2878$$

**Rezultat:** probabilitatea este de aproximativ **28.78%**.

---

**2. Probabilitatea ca pacientul sa aiba boala, stiind ca are tuse si dificultati de respiratie**

Aplicam regula lui Bayes:

$$P(B=1\mid T=1,D=1)=\frac{P(B=1,T=1,D=1)}{P(T=1,D=1)}$$

Pentru pacientii cu boala:

$$P(B=1,T=1,D=1)=0.1\cdot0.8\cdot0.9=0.072$$

Pentru pacientii fara boala:

$$P(B=0,T=1,D=1)=0.9\cdot0.3\cdot0.5=0.135$$

Adunam cele doua cazuri pentru a obtine probabilitatea de a avea simultan tuse si dificultati de respiratie:

$$P(T=1,D=1)=0.072+0.135=0.207$$

Aplicam regula lui Bayes:

$$P(B=1\mid T=1,D=1)=\frac{0.072}{0.207}\approx0.3478$$

**Rezultat:** probabilitatea este de aproximativ **34.78%**.

## Exercițiul 3 – Joc între doi jucători

Jocul se desfășoară între doi jucători, \(J_0\) și \(J_1\).

La început, se aruncă o monedă normală pentru a decide cine începe. În prima rundă, jucătorul desemnat aruncă un zar normal și obține numărul \(n\). În a doua rundă, celălalt jucător aruncă propria monedă de \(2n\) ori, iar \(m\) reprezintă numărul de steme obținute.

Jucătorul din prima rundă câștigă dacă \(n \geq m\). În caz contrar, câștigă jucătorul din a doua rundă.

Moneda jucătorului \(J_0\) este normală, iar moneda jucătorului \(J_1\) este măsluită, având probabilitatea de a obține stema egală cu \(\frac{4}{7}\). Ambele zaruri sunt normale.

### a) Estimarea șanselor de câștig prin simulare

Simulăm jocul de 10.000 de ori pentru a estima care dintre cei doi jucători are șansele cele mai mari de câștig.

### Metoda de simulare

Pentru a estima șansele de câștig ale celor doi jucători, simulăm jocul de 10.000 de ori.

La fiecare joc:

1. Stabilim cine începe, \(J_0\) sau \(J_1\), folosind o monedă normală.
2. Jucătorul care începe aruncă un zar normal și obține numărul \(n\), cu valori posibile de la 1 la 6.
3. Celălalt jucător aruncă moneda proprie de \(2n\) ori. Pentru moneda lui \(J_0\), probabilitatea de stemă este \(\frac12\), iar pentru moneda măsluită a lui \(J_1\), probabilitatea de stemă este \(\frac47\).
4. Numărăm stemele obținute și notăm rezultatul cu \(m\).
5. Dacă \(n \geq m\), câștigă jucătorul din prima rundă. În caz contrar, câștigă celălalt jucător.

La final, numărăm victoriile fiecărui jucător și împărțim numărul acestora la 10.000 pentru a estima probabilitatea de câștig.

In [16]:

import numpy as np

nr_jocuri = 10000
castiguri_J0 = 0
castiguri_J1 = 0

for _ in range(nr_jocuri):
    # stabilim cine incepe
    primul = np.random.choice([0, 1])

    # jucatorul din prima runda arunca zarul
    n = np.random.randint(1, 7)

    # stabilim moneda celui de-al doilea jucator
    al_doilea = 1 - primul

    if al_doilea == 0:
        p_stema = 1 / 2
    else:
        p_stema = 4 / 7

    # al 2lea jucator arunca moneda de 2n ori
    m = np.random.binomial(2 * n, p_stema)

    # stabilim castigatorul
    if n >= m:
        castigator = primul
    else:
        castigator = al_doilea

    if castigator == 0:
        castiguri_J0 += 1
    else:
        castiguri_J1 += 1

print("Castiguri J0:", castiguri_J0)
print("Castiguri J1:", castiguri_J1)

print("Probabilitate estimata J0:",
      castiguri_J0 / nr_jocuri)

print("Probabilitate estimata J1:",
      castiguri_J1 / nr_jocuri)


Castiguri J0: 4330
Castiguri J1: 5670
Probabilitate estimata J0: 0.433
Probabilitate estimata J1: 0.567


### Concluzie

În urma simulării celor 10.000 de jocuri, jucătorul \(J_0\) a câștigat de **4330** ori, iar jucătorul \(J_1\) de **5670** ori.

Probabilitatea estimată de câștig pentru \(J_0\) este **0.433 (43.3%)**, iar pentru \(J_1\) este **0.567 (56.7%)**.

Conform rezultatelor simulării, jucătorul cu probabilitatea estimată de câștig mai mare este **\(J_1\)**.

### b) Construirea retelei bayesiene

Pentru a descrie jocul folosind o retea bayesiana, definim urmatoarele variabile aleatoare:

- \(J\): jucatorul care incepe jocul (\(J=0\) pentru \(J_0\), \(J=1\) pentru \(J_1\));
- \(N\): numarul obtinut la aruncarea zarului de catre primul jucator;
- \(M\): numarul de steme obtinute de al doilea jucator, care arunca moneda de \(2N\) ori;
- \(W\): castigatorul jocului (\(W=0\) pentru \(J_0\), \(W=1\) pentru \(J_1\)).

Jucatorul care incepe este ales cu probabilitati egale, deoarece moneda folosita pentru stabilirea primului jucator este normala.

Numarul \(N\) poate lua valorile de la 1 la 6, fiecare cu probabilitatea \(1/6\), deoarece zarul este normal.

Variabila \(M\) depinde de jucatorul care arunca moneda si de numarul \(N\). Moneda lui \(J_0\) are probabilitatea de stema \(1/2\), iar moneda lui \(J_1\) are probabilitatea de stema \(4/7\). Moneda este aruncata de \(2N\) ori.

Castigatorul este determinat comparand \(N\) si \(M\): daca \(N \geq M\), castiga jucatorul din prima runda; in caz contrar, castiga celalalt jucator.

In [17]:

from pgmpy.models import DiscreteBayesianNetwork

model3 = DiscreteBayesianNetwork([
    ("J", "M"),
    ("N", "M"),
    ("J", "W"),
    ("N", "W"),
    ("M", "W")
])

print(model3.nodes())
print(model3.edges())


['J', 'M', 'N', 'W']
[('J', 'M'), ('J', 'W'), ('M', 'W'), ('N', 'M'), ('N', 'W')]


In [22]:
import numpy as np
from math import comb
from pgmpy.factors.discrete import TabularCPD

# J: jucatorul care incepe
cpd_J = TabularCPD(
    variable="J",
    variable_card=2,
    values=[[0.5], [0.5]],
    state_names={"J":[0,1]}
)
# N: nr obtinut la zar
cpd_N = TabularCPD(
    variable="N",
    variable_card=6,
    values=[[1/6], [1/6], [1/6], [1/6], [1/6], [1/6]],
    state_names={"N":[1,2,3,4,5,6]}
)
# M: nr de steme obtinute la al 2lea jucator
valori_M=[]

for j in [0,1]:
  for n in [1,2,3,4,5,6]:
    al_doilea = 1 - j

    if al_doilea == 0:
      p = 1 / 2
    else:
      p = 4 / 7

    coloana = []
    for m in range (13):
      if m <= 2 * n:
        prob = comb(2*n, m) * p**m * (1-p)**(2*n-m)
      else:
        prob = 0
      coloana.append(prob)

    valori_M.append(coloana)

cpd_M = TabularCPD(
    variable="M",
    variable_card=13,
    values=np.array(valori_M).T.tolist(),
    evidence=["J", "N"],
    evidence_card=[2, 6],
    state_names={
        "M": list(range(13)),
        "J": [0, 1],
        "N": [1, 2, 3, 4, 5, 6]
    }
)

# W: castigatorul
valori_W = [[],[]]

for j in [0,1]:
  for n in [1,2,3,4,5,6]:
    for m in range(13):
      if n >= m:
        castigator = j
      else:
        castigator = 1 - j

      valori_W[0].append(1 if castigator == 0 else 0)
      valori_W[1].append(1 if castigator == 1 else 0)

cpd_W = TabularCPD(
    variable="W",
    variable_card=2,
    values=valori_W,
    evidence=["J", "N", "M"],
    evidence_card=[2, 6, 13],
    state_names={
        "W": [0, 1],
        "J": [0, 1],
        "N": [1, 2, 3, 4, 5, 6],
        "M": list(range(13))
    }
)

model3.add_cpds(cpd_J, cpd_N, cpd_M, cpd_W)

print("Modelul este corect:", model3.check_model())

Modelul este corect: True


In [24]:

from pgmpy.inference import VariableElimination

infer3 = VariableElimination(model3)

rezultat = infer3.query(variables=["W"])
print(rezultat)

+------+----------+
| W    |   phi(W) |
+======+==========+
| W(0) |   0.4267 |
+------+----------+
| W(1) |   0.5733 |
+------+----------+


### Inferenta asupra castigatorului

Folosim reteaua bayesiana pentru a determina probabilitatile initiale ale castigatorului. Rezultatul obtinut prin inferenta indica probabilitatea ca fiecare jucator sa castige jocul, tinand cont de regulile si probabilitatile definite in model.

### c) Determinarea jucatorului care a inceput jocul

Stim ca in a2a runda s-a obtinut o singura stema adica (M = 1)

Dorim sa detemrinam care jucator este cel mai probabil sa fi inceput jocul, calculand prob. cond.

$$
P(J=0\mid M=1)
$$
$$
P(J=1\mid M=1)
$$
Folosim reteaua bayesiana definita la pct. b) , introducand (M = 1) ca evidenta. Comparand cele 2 prob., putem detemrina care jcator este cel mai probabil sa fi inceput jocul.

In [26]:
rezultat_c = infer3.query(
    variables=["J"],
    evidence={"M": 1}
)

print(rezultat_c)


+------+----------+
| J    |   phi(J) |
+======+==========+
| J(0) |   0.4529 |
+------+----------+
| J(1) |   0.5471 |
+------+----------+


###Concluzie

Deoarece (0.5471 > 0.4529), (J_1) este cel mai probabil sa fi inceput jocul , stiind ca in a2a runda s-a obtinut o singura stema.